# Supplier ETL

## 2.1 Supplier Source Data

The Supplier data is simulated as an extract from the Procurement / ERP source system.

The raw source data will be inspected before any cleaning, transformation, validation, or PostgreSQL loading is performed.

The purpose of this stage is to understand the source structure, including:

- available columns
- data types
- row count
- missing values
- duplicate records
- formatting inconsistencies
- potential data-quality issues

No changes will be made to the raw source file or PostgreSQL during this stage.

## 2.2 Extract Raw Supplier Data

The Supplier raw data is extracted from the simulated Procurement / ERP source system into a Pandas DataFrame.

At this stage, the source data is only being read into Python. No cleaning, transformation, or modification is performed.

The extracted data will be used for the following profiling and ETL steps.

In [ ]:
# Import Pandas so we can work with tabular data
import pandas as pd

In [ ]:
# Read the raw Supplier CSV into a Pandas DataFrame
supplier_raw = pd.read_csv(
    "../data/02_Procurement_ERP/procurement_supplier_master.csv"
)

In [38]:
from pathlib import Path
import sys

project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from profiler.data_profiler_v1 import profile_dataset

In [39]:
import importlib
import profiler.data_profiler_v1 as profiler

importlib.reload(profiler)

<module 'profiler.data_profiler_v1' from 'c:\\JEP\\DATA ANALYST PORTFOLIO\\Lj_Dev_Commerce\\profiler\\data_profiler_v1.py'>

In [40]:
profile_results = profile_dataset(supplier_raw)

In [ ]:
# Display the first 5 rows to confirm the Supplier data was loaded correctly
supplier_raw.head()

## 2.3 Raw Data Profiling

The raw Supplier dataset is profiled systematically before any cleaning or transformation.

Each column is reviewed according to its role and expected data behavior. The profiling checks for data-quality issues such as missing values, blank values, duplicates, formatting inconsistencies, invalid values, unexpected patterns, and business-rule violations where applicable.

The purpose is to understand the source data completely before defining cleaning and transformation rules.

### 2.3.1 Dataset Overview

Before profiling individual columns, the overall structure of the raw Supplier dataset is reviewed.

The objective is to establish the basic characteristics of the source data, including its size, column structure, and general data types.

In [ ]:
# Check the number of records and columns in the raw Supplier dataset
supplier_raw.shape

### 2.3.2 Column Structure

The column structure of the raw Supplier dataset is reviewed to identify all fields received from the source system before assessing individual column quality.

In [ ]:
# Display all column names received from the source system
supplier_raw.columns

### 2.3.3 Data Types

The data type detected for each raw Supplier column is reviewed to identify fields that may require type validation or transformation later in the ETL process.

In [ ]:
# Check the data type detected by Pandas for each Supplier column
supplier_raw.dtypes

### 2.3.4 Missing Values

The raw Supplier dataset is checked for missing values in each column.

This identifies fields where information is absent and may require a business rule or treatment during the cleaning stage.

In [ ]:
# Count missing values in each Supplier column
supplier_raw.isna().sum()

### 2.3.5 Complete Duplicate Rows

The raw Supplier dataset is checked for records where all column values are identical.

This identifies exact duplicate records before individual column profiling and later cleaning.

In [ ]:
# Count rows that are completely identical to an earlier row
supplier_raw.duplicated().sum()

### 2.3.6 Blank and Whitespace Values

Text-based fields are checked for empty strings and whitespace-only values.

This complements the missing-value check because a field can contain a string value that is technically not missing but is still effectively blank.

In [ ]:
# Count blank or whitespace-only values across text-based Supplier columns
supplier_raw.select_dtypes(include="object").apply(
    lambda col: col.astype(str).str.strip().eq("").sum()
)

### 2.3.7 SupplierCode Profiling

SupplierCode is treated as the business identifier for the Supplier record.

The field is profiled for completeness, uniqueness, formatting consistency, unexpected values, and identifier integrity before it is mapped to the target supplier_id field.

In [ ]:
# Check for missing SupplierCode values
supplier_raw["SupplierCode"].isna().sum()

In [ ]:
# Check for SupplierCode values that are empty or contain only spaces
supplier_raw["SupplierCode"].astype(str).str.strip().eq("").sum()

In [ ]:
# Count SupplierCode values with unwanted spaces before or after the ID
(
    supplier_raw["SupplierCode"].astype(str)
    != supplier_raw["SupplierCode"].astype(str).str.strip()
).sum()

In [ ]:
# Display all SupplierCode values to inspect their actual format
supplier_raw["SupplierCode"].tolist()

In [ ]:
# Check the character length of each SupplierCode value
supplier_raw["SupplierCode"].str.len().value_counts()

In [ ]:
# Check whether every SupplierCode follows the expected SUP + 3 digits pattern
supplier_raw["SupplierCode"].str.fullmatch(r"SUP\d{3}").all()

In [ ]:
# Check the number of SupplierCode values containing characters outside A-Z and 0-9
supplier_raw["SupplierCode"].str.contains(r"[^A-Z0-9]", regex=True).sum()

In [ ]:
# Count the number of unique SupplierCode values
supplier_raw["SupplierCode"].nunique()

In [ ]:
# Count how many SupplierCode values occur more than once
supplier_raw["SupplierCode"].value_counts().loc[lambda x: x > 1]

In [ ]:
# Show how many times each SupplierCode appears in the dataset
supplier_raw["SupplierCode"].value_counts().sort_index()

In [ ]:
# Check for common placeholder values that may represent missing SupplierCode data
supplier_raw["SupplierCode"].str.strip().str.upper().isin(
    ["NULL", "N/A", "NA", "UNKNOWN", "TBD", "-"]
).sum()

In [ ]:
# Extract the numeric portion of each SupplierCode for sequence profiling
supplier_code_numbers = (
    supplier_raw["SupplierCode"]
    .str.extract(r"SUP(\d{3})")[0]
    .astype(int)
)

supplier_code_numbers.sort_values().tolist()

In [ ]:
# Check whether any SupplierCode contains lowercase letters
supplier_raw["SupplierCode"].str.contains(r"[a-z]", regex=True).sum()

In [ ]:
# Display all records where SupplierCode appears more than once
supplier_raw[
    supplier_raw["SupplierCode"].duplicated(keep=False)
].sort_values("SupplierCode")

### 2.3.8 SupplierName Profiling

SupplierName is profiled as a descriptive text field.

The field is reviewed for completeness, blank values, whitespace issues, value consistency, duplicate or repeated names, formatting, unusual characters, and suspicious values before cleaning and transformation.

In [ ]:
# Check for missing SupplierName values
supplier_raw["SupplierName"].isna().sum()

In [ ]:
# Check for SupplierName values that are empty or contain only spaces
supplier_raw["SupplierName"].astype(str).str.strip().eq("").sum()

In [ ]:
# Count SupplierName values with unwanted spaces before or after the name
(
    supplier_raw["SupplierName"].astype(str)
    != supplier_raw["SupplierName"].astype(str).str.strip()
).sum()

In [ ]:
# Display SupplierName values that contain leading or trailing spaces
supplier_raw[
    supplier_raw["SupplierName"].astype(str)
    != supplier_raw["SupplierName"].astype(str).str.strip()
][["SupplierCode", "SupplierName"]]

In [ ]:
# Display all SupplierName values to inspect names and their formatting
supplier_raw["SupplierName"].tolist()

In [ ]:
# Count the number of unique SupplierName values
supplier_raw["SupplierName"].nunique()

In [ ]:
# Count SupplierName values that occur more than once
supplier_raw["SupplierName"].value_counts().loc[lambda x: x > 1]

In [ ]:
# Display all records that share the same SupplierName
supplier_raw[
    supplier_raw["SupplierName"].eq("Nova Tech Distribution")
]

In [ ]:
# Check the capitalization pattern of each SupplierName value
supplier_raw["SupplierName"].str.isupper().value_counts()

In [ ]:
# Check how many SupplierName values follow title-case formatting
supplier_raw["SupplierName"].str.istitle().value_counts()

In [ ]:
# Display SupplierName values that are not in title-case format
supplier_raw[
    ~supplier_raw["SupplierName"].str.istitle()
][["SupplierCode", "SupplierName"]]

In [ ]:
# Check whether any SupplierName values are entirely lowercase
supplier_raw["SupplierName"].str.islower().value_counts()

In [ ]:
# Count SupplierName values containing characters outside letters, numbers, spaces, and common punctuation
supplier_raw["SupplierName"].str.contains(
    r"[^A-Za-z0-9\s&.,'()-]",
    regex=True
).sum()

In [ ]:
# Check the character length of each SupplierName
supplier_raw["SupplierName"].str.len().describe()

In [ ]:
# Display the shortest and longest SupplierName values for inspection
supplier_name_lengths = supplier_raw["SupplierName"].str.len()

supplier_raw.loc[
    supplier_name_lengths.isin(
        [supplier_name_lengths.min(), supplier_name_lengths.max()]
    ),
    ["SupplierCode", "SupplierName"]
].sort_values("SupplierName")

In [ ]:
# Check for common placeholder values that may represent missing SupplierName data
supplier_raw["SupplierName"].str.strip().str.upper().isin(
    ["NULL", "N/A", "NA", "UNKNOWN", "TBD", "-"]
).sum()

In [ ]:
# Check whether removing outer spaces creates duplicate SupplierName values
supplier_raw["SupplierName"].str.strip().value_counts().loc[lambda x: x > 1]

### 2.3.9 ContactPerson Profiling

ContactPerson is profiled as a person-name text field.

The field is reviewed for completeness, blank values, whitespace issues, value consistency, duplicate or repeated contact names, capitalization, unusual characters, suspicious values, and text length before cleaning and transformation.

In [ ]:
# Check for missing ContactPerson values
supplier_raw["ContactPerson"].isna().sum()

In [ ]:
# Check for ContactPerson values that are empty or contain only spaces
supplier_raw["ContactPerson"].astype(str).str.strip().eq("").sum()

In [ ]:
# Count ContactPerson values with unwanted spaces before or after the name
(
    supplier_raw["ContactPerson"].astype(str)
    != supplier_raw["ContactPerson"].astype(str).str.strip()
).sum()

In [ ]:
# Display all ContactPerson values to inspect their actual content and formatting
supplier_raw["ContactPerson"].tolist()

In [ ]:
# Count the number of unique ContactPerson values
supplier_raw["ContactPerson"].nunique()

In [ ]:
# Count ContactPerson values that occur more than once
supplier_raw["ContactPerson"].value_counts().loc[lambda x: x > 1]

In [ ]:
# Display all records that share the same ContactPerson
supplier_raw[
    supplier_raw["ContactPerson"].eq("Daniel Cruz")
]

In [ ]:
# Check whether any ContactPerson values are entirely uppercase
supplier_raw["ContactPerson"].str.isupper().value_counts()

In [ ]:
# Check how many ContactPerson values follow title-case formatting
supplier_raw["ContactPerson"].str.istitle().value_counts()

In [ ]:
# Count ContactPerson values containing unexpected characters
supplier_raw["ContactPerson"].str.contains(
    r"[^A-Za-z\s.'-]",
    regex=True
).sum()

In [ ]:
# Check the character length distribution of ContactPerson values
supplier_raw["ContactPerson"].str.len().describe()

In [ ]:
# Display the shortest and longest ContactPerson values for inspection
contact_lengths = supplier_raw["ContactPerson"].str.len()

supplier_raw.loc[
    contact_lengths.isin(
        [contact_lengths.min(), contact_lengths.max()]
    ),
    ["SupplierCode", "ContactPerson"]
].sort_values("ContactPerson")

In [ ]:
# Check for common placeholder values that may represent missing ContactPerson data
supplier_raw["ContactPerson"].str.strip().str.upper().isin(
    ["NULL", "N/A", "NA", "UNKNOWN", "TBD", "-"]
).sum()

In [ ]:
# Check whether trimming spaces and standardizing case reveals repeated ContactPerson values
supplier_raw["ContactPerson"].str.strip().str.casefold().value_counts().loc[lambda x: x > 1]

### 2.3.10 EmailAddress Profiling

EmailAddress is profiled as a contact-information field.

The field is reviewed for completeness, blank values, whitespace issues, duplicate or repeated email addresses, formatting and structural validity, capitalization, unusual characters, placeholder values, and consistency before cleaning and transformation.

In [ ]:
# Check for missing EmailAddress values
supplier_raw["EmailAddress"].isna().sum()

In [ ]:
# Check for EmailAddress values that are empty or contain only spaces
supplier_raw["EmailAddress"].astype(str).str.strip().eq("").sum()

In [ ]:
# Check for EmailAddress values with unwanted spaces before or after the email
(
    supplier_raw["EmailAddress"].astype(str)
    != supplier_raw["EmailAddress"].astype(str).str.strip()
).sum()

In [ ]:
# Display all EmailAddress values to inspect their actual content and formatting
supplier_raw["EmailAddress"].tolist()

In [ ]:
# Count the number of unique EmailAddress values
supplier_raw["EmailAddress"].nunique()

In [ ]:
# Count EmailAddress values that occur more than once
supplier_raw["EmailAddress"].value_counts().loc[lambda x: x > 1]

In [ ]:
# Display all records that share the same EmailAddress
supplier_raw[
    supplier_raw["EmailAddress"].eq("nova.tech@email.com")
]

In [ ]:
# Check whether every EmailAddress contains exactly one @ character
(supplier_raw["EmailAddress"].str.count("@") == 1).value_counts()

In [ ]:
# Check for EmailAddress values with no text before or after the @ symbol
email_parts = supplier_raw["EmailAddress"].str.split("@")

(
    email_parts.str[0].eq("")
    | email_parts.str[1].eq("")
).sum()

In [ ]:
# Extract the domain portion of each EmailAddress for profiling
supplier_raw["EmailAddress"].str.split("@").str[1].value_counts()

In [ ]:
# Separate the local part of each email to inspect its values and frequency
supplier_raw["EmailAddress"].str.split("@").str[0].value_counts()

In [ ]:
# Check whether every email domain contains a dot separating the domain name and extension
email_domains = supplier_raw["EmailAddress"].str.split("@").str[1]

email_domains.str.contains(r"\.", regex=True).value_counts()

In [ ]:
# Check for email domains with a leading dot, trailing dot, or consecutive dots
email_domains = supplier_raw["EmailAddress"].str.split("@").str[1]

(
    email_domains.str.startswith(".")
    | email_domains.str.endswith(".")
    | email_domains.str.contains("..", regex=False)
).sum()

In [ ]:
# Check for email local parts with a leading dot, trailing dot, or consecutive dots
email_local = supplier_raw["EmailAddress"].str.split("@").str[0]

(
    email_local.str.startswith(".")
    | email_local.str.endswith(".")
    | email_local.str.contains("..", regex=False)
).sum()

In [ ]:
# Check for EmailAddress values containing characters outside a reasonable email character set
supplier_raw["EmailAddress"].str.contains(
    r"[^A-Za-z0-9.!#$%&'*+/=?^_`{|}~@-]",
    regex=True,
    na=False
).sum()

In [ ]:
# Check whether any EmailAddress values are entirely uppercase
supplier_raw["EmailAddress"].str.isupper().value_counts()

In [ ]:
# Check whether any EmailAddress values are entirely lowercase
supplier_raw["EmailAddress"].str.islower().value_counts()

In [ ]:
# Check for common placeholder values that may represent missing EmailAddress data
supplier_raw["EmailAddress"].str.strip().str.upper().isin(
    ["NULL", "N/A", "NA", "UNKNOWN", "TBD", "-"]
).sum()

In [ ]:
# Check the character length distribution of EmailAddress values
supplier_raw["EmailAddress"].str.len().describe()

In [ ]:
# Display the shortest and longest EmailAddress values for inspection
email_lengths = supplier_raw["EmailAddress"].str.len()

supplier_raw.loc[
    email_lengths.isin(
        [email_lengths.min(), email_lengths.max()]
    ),
    ["SupplierCode", "EmailAddress"]
].sort_values("EmailAddress")

In [ ]:
# Check for duplicate emails after ignoring outer spaces and capitalization differences
supplier_raw["EmailAddress"].str.strip().str.casefold().value_counts().loc[lambda x: x > 1]

In [ ]:
# Check whether each EmailAddress follows a basic practical email format
email_pattern = r"^[A-Za-z0-9.!#$%&'*+/=?^_`{|}~-]+@[A-Za-z0-9-]+(?:\.[A-Za-z0-9-]+)+$"

supplier_raw["EmailAddress"].str.match(
    email_pattern,
    na=False
).value_counts()

#### 2.3.11.1 Automated Missing-Value Profiling

The first automated profiling check evaluates missing values across all columns.

This provides a quick completeness assessment of the dataset before performing more detailed, column-specific checks.

In [ ]:
# Profile missing values across all columns
missing_profile = supplier_raw.isna().sum()

missing_profile

In [ ]:
# Calculate the percentage of missing values in each column
missing_percentage = (supplier_raw.isna().mean() * 100).round(2)

missing_percentage

#### 2.3.11.2 Automated Exact-Duplicate Profiling

The next automated check identifies records that are completely duplicated across all columns.

Exact duplicate rows are investigated separately from duplicated values within individual columns because a repeated identifier or value does not necessarily mean that the entire record is duplicated.

In [ ]:
# Count records that are exact duplicates across all columns
exact_duplicate_count = supplier_raw.duplicated().sum()

exact_duplicate_count

In [ ]:
# Display the records identified as exact duplicates
supplier_raw[supplier_raw.duplicated(keep=False)]

In [ ]:
# Count duplicated values in each column to identify fields that may require investigation
duplicate_value_counts = supplier_raw.apply(
    lambda column: column.duplicated().sum()
)

duplicate_value_counts

#### 2.3.11.3 Automated Data-Type Profiling

The next automated check identifies the data type stored in each column.

Data types help determine which additional profiling, validation, and transformation checks are appropriate for each field.

In [ ]:
# Profile the stored data type of every column
data_type_profile = supplier_raw.dtypes

data_type_profile

#### 2.3.11.4 Automated Unique-Value Profiling

The next automated check measures the number of distinct values in each column.

Unique-value counts help identify identifier-like fields, categorical fields, and low- or high-cardinality columns that may require different profiling and validation approaches.

In [ ]:
# Count distinct values in every column
unique_value_profile = supplier_raw.nunique(dropna=True)

unique_value_profile

#### 2.3.11.5 Automated Dataset Size Profiling

The dataset size is recorded as the total number of rows and columns received.

This provides a baseline for validating record counts before and after cleaning and transformation.

In [ ]:
# Record the number of rows and columns in the dataset
row_count, column_count = supplier_raw.shape

print("Rows:", row_count)
print("Columns:", column_count)

#### 2.3.11.6 Automated Whitespace Profiling

The next automated check identifies text values with leading or trailing whitespace.

Unwanted whitespace can create false differences during matching, grouping, filtering, and duplicate detection.

In [ ]:
# Identify text columns containing values with leading or trailing whitespace
text_columns = supplier_raw.select_dtypes(include="object").columns

whitespace_profile = {
    column: (
        supplier_raw[column].astype(str)
        != supplier_raw[column].astype(str).str.strip()
    ).sum()
    for column in text_columns
}

pd.Series(whitespace_profile)

In [ ]:
# Display text values with leading or trailing whitespace for investigation
for column in text_columns:
    mask = (
        supplier_raw[column].astype(str)
        != supplier_raw[column].astype(str).str.strip()
    )

    if mask.any():
        print(f"\n{column}:")
        print(supplier_raw.loc[mask, [column]])

#### 2.3.11.7 Automated Text-Length Profiling

The next automated check profiles the character length of values in text-like columns.

Length statistics provide an initial indication of unusually short or long values that may require further investigation.

In [ ]:
# Profile the character length of values in each text-like column
text_length_profile = pd.DataFrame({
    column: supplier_raw[column].astype(str).str.len().describe()
    for column in text_columns
})

text_length_profile

#### 2.3.11.8 Date-Like Column Profiling

The next automated check identifies object columns whose values can be interpreted as dates.

This helps detect date-like fields that are stored as text and determines whether their values can be converted reliably during the transformation stage.

In [ ]:
# Identify columns that appear to represent dates based on their column names
date_candidates = [
    column for column in supplier_raw.columns
    if any(keyword in column.lower() for keyword in ["date", "time", "createdon", "modifiedon"])
]

date_candidates

In [ ]:
# Check how many values in each date candidate can be converted to valid dates
date_validity_profile = {
    column: pd.to_datetime(
        supplier_raw[column],
        errors="coerce"
    ).notna().sum()
    for column in date_candidates
}

pd.Series(date_validity_profile)

In [ ]:
# Profile the earliest and latest dates in each date candidate
date_range_profile = {}

for column in date_candidates:
    converted_dates = pd.to_datetime(
        supplier_raw[column],
        errors="coerce"
    )

    date_range_profile[column] = {
        "MinimumDate": converted_dates.min(),
        "MaximumDate": converted_dates.max()
    }

pd.DataFrame(date_range_profile).T

In [ ]:
# Check for CreatedOn and ModifiedOn values that occur after today's date
today = pd.Timestamp.today().normalize()

future_date_profile = {}

for column in date_candidates:
    converted_dates = pd.to_datetime(
        supplier_raw[column],
        errors="coerce"
    )

    future_date_profile[column] = (
        converted_dates > today
    ).sum()

pd.Series(future_date_profile)

In [ ]:
# Inspect the raw values in date candidate columns to identify their source date format
for column in date_candidates:
    print(f"\n{column}:")
    print(supplier_raw[column].tolist())

In [ ]:
# Check for records where ModifiedOn occurs before CreatedOn
created_dates = pd.to_datetime(supplier_raw["CreatedOn"], errors="coerce")
modified_dates = pd.to_datetime(supplier_raw["ModifiedOn"], errors="coerce")

(modified_dates < created_dates).sum()

### 2.3.12 Consolidated Automated Data Profiler

The individual profiling checks developed above are consolidated into a reusable profiling function.

The function provides a first-pass assessment of dataset size, completeness, uniqueness, duplicates, data types, whitespace, and text length.

The profiler identifies potential data-quality issues for further investigation; it does not make business decisions automatically.

In [ ]:
def profile_dataset(df):
    """
    Generate a general automated data-quality profile.
    """

    profile = pd.DataFrame({
        "DataType": df.dtypes.astype(str),
        "MissingCount": df.isna().sum(),
        "MissingPercent": (df.isna().mean() * 100).round(2),
        "UniqueCount": df.nunique(dropna=True),
        "DuplicateValueCount": df.apply(lambda col: col.duplicated().sum())
    })

    # Check leading/trailing whitespace for text-like columns
    text_columns = df.select_dtypes(include="object").columns

    whitespace_counts = {}

    for column in text_columns:
        whitespace_counts[column] = (
            df[column].astype(str)
            != df[column].astype(str).str.strip()
        ).sum()

    profile["WhitespaceCount"] = pd.Series(whitespace_counts)

    # Exact duplicate rows
    exact_duplicate_count = df.duplicated().sum()

    print(f"Rows: {df.shape[0]}")
    print(f"Columns: {df.shape[1]}")
    print(f"Exact duplicate rows: {exact_duplicate_count}")

    return profile

In [ ]:
supplier_profile = profile_dataset(supplier_raw)

supplier_profile

### 2.3.13 Specialized Date Profiling

Date profiling is performed on columns identified as date-like. The profiler checks date validity, date range, future dates, and basic chronological relationships where applicable.

In [ ]:
def profile_date_columns(df):
    """
    Profile date-like columns and identify:
    - valid dates
    - invalid dates
    - ambiguous dates
    - mixed date formats
    - date range
    - future dates
    """

    date_candidates = [
        column for column in df.columns
        if any(
            keyword in column.lower()
            for keyword in ["date", "time", "createdon", "modifiedon"]
        )
    ]

    results = []

    for column in date_candidates:

        raw_values = df[column].dropna().astype(str).str.strip()

        # Try common date formats separately
        parsed_iso = pd.to_datetime(
            raw_values,
            format="%Y-%m-%d",
            errors="coerce"
        )

        parsed_dmy = pd.to_datetime(
            raw_values,
            format="%d/%m/%Y",
            errors="coerce"
        )

        parsed_mdy = pd.to_datetime(
            raw_values,
            format="%m/%d/%Y",
            errors="coerce"
        )

        # General parsing
        parsed_general = pd.to_datetime(
            raw_values,
            errors="coerce"
        )

        valid_count = parsed_general.notna().sum()
        invalid_count = parsed_general.isna().sum()

        # Detect values that can be interpreted under both DMY and MDY
        ambiguous_mask = (
            parsed_dmy.notna()
            & parsed_mdy.notna()
            & (parsed_dmy != parsed_mdy)
        )

        ambiguous_count = ambiguous_mask.sum()

        # Detect whether values match different recognized formats
        format_matches = pd.DataFrame({
            "ISO": parsed_iso.notna(),
            "DMY": parsed_dmy.notna(),
            "MDY": parsed_mdy.notna()
        })

        formats_used = format_matches.sum()
        mixed_format = (formats_used > 0).sum() > 1

        results.append({
            "Column": column,
            "ValidDateCount": valid_count,
            "InvalidDateCount": invalid_count,
            "AmbiguousDateCount": ambiguous_count,
            "MixedFormat": mixed_format,
            "MinimumDate": parsed_general.min(),
            "MaximumDate": parsed_general.max(),
            "FutureDateCount": (
                parsed_general > pd.Timestamp.today().normalize()
            ).sum()
        })

    return pd.DataFrame(results)

In [ ]:
date_profile = profile_date_columns(supplier_raw)

date_profile

### 2.3.14 Reusable Data Profiling Framework

A reusable, dataset-independent profiling framework designed to perform a first-pass assessment of data quality across different datasets.

The framework combines general profiling with data-type and pattern-based checks. It identifies potential issues and provides evidence for investigation rather than automatically making business decisions.

The framework is designed to be adaptable to different source systems, date conventions, locales, and business rules.

In [ ]:
def profile_dataset(df, config=None):
    """
    Reusable data-quality profiling framework.

    Parameters
    ----------
    df : pandas.DataFrame
        Dataset to profile.

    config : dict, optional
        Optional source/business-specific configuration.

    Returns
    -------
    dict
        Profiling results grouped by profiling area.
    """

    if config is None:
        config = {}

    results = {}

    # -------------------------------------------------
    # 1. GENERAL PROFILING
    # -------------------------------------------------

    results["general"] = {
        "rows": len(df),
        "columns": len(df.columns),
        "missing": df.isna().sum(),
        "missing_percent": (df.isna().mean() * 100).round(2),
        "data_types": df.dtypes.astype(str),
        "unique_counts": df.nunique(dropna=True),
        "exact_duplicate_rows": int(df.duplicated().sum())
    }

    # -------------------------------------------------
    # 2. TEXT PROFILING
    # -------------------------------------------------

    text_columns = df.select_dtypes(
        include=["object", "string"]
    ).columns

    results["text"] = {
        "text_columns": list(text_columns)
    }

    # -------------------------------------------------
    # 3. CATEGORICAL / BOOLEAN PROFILING
    # -------------------------------------------------

    categorical_columns = df.select_dtypes(
        include=["object", "string", "category", "bool"]
    ).columns

    results["categorical"] = {
        "columns": list(categorical_columns),
        "unique_counts": df[categorical_columns].nunique(dropna=True)
    }

    # -------------------------------------------------
    # 4. NUMERIC PROFILING
    # -------------------------------------------------

    numeric_columns = df.select_dtypes(
        include="number"
    ).columns

    results["numeric"] = {
        "numeric_columns": list(numeric_columns)
    }

    # -------------------------------------------------
    # 5. DATE / TIME PROFILING
    # -------------------------------------------------

    results["date"] = {
        "date_columns": []
    }

    # -------------------------------------------------
    # 6. PATTERN PROFILING
    # -------------------------------------------------

    results["patterns"] = {
        "email_columns": [],
        "phone_columns": [],
        "identifier_columns": []
    }

    # -------------------------------------------------
    # 7. CONFIGURATION
    # -------------------------------------------------

    results["configuration"] = config

    return results

In [ ]:
profile_results = profile_dataset(supplier_raw)

profile_results.keys()

In [ ]:
def profile_dataset(df, config=None):
    """
    Reusable data-quality profiling framework.

    The profiler:
    1. Detects likely semantic field types.
    2. Profiles the data using type-appropriate checks.
    3. Flags potential issues.
    4. Prioritizes issues by severity.

    It does not automatically clean or make business decisions.
    """

    if config is None:
        config = {}

    results = {}

    # -------------------------------------------------
    # 1. SEMANTIC FIELD-TYPE DETECTION
    # -------------------------------------------------

    detected_types = detect_field_types(df)

    results["field_types"] = detected_types

    # -------------------------------------------------
    # 2. GENERAL PROFILING
    # -------------------------------------------------

    general_profile = pd.DataFrame({
        "DataType": df.dtypes.astype(str),
        "DetectedFieldType": detected_types,
        "MissingCount": df.isna().sum(),
        "MissingPercent": (
            df.isna().mean() * 100
        ).round(2),
        "UniqueCount": df.nunique(dropna=True),
        "DuplicateValueCount": df.apply(
            lambda col: col.duplicated().sum()
        )
    })

    results["general"] = {
        "rows": len(df),
        "columns": len(df.columns),
        "exact_duplicate_rows": int(
            df.duplicated().sum()
        ),
        "column_profile": general_profile
    }

    # -------------------------------------------------
    # 3. TEXT PROFILING
    # -------------------------------------------------

    text_columns = detected_types[
        detected_types == "text"
    ].index

    text_profile = {}

    for column in text_columns:

        values = (
            df[column]
            .dropna()
            .astype(str)
        )

        stripped_values = values.str.strip()

        text_profile[column] = {
            "WhitespaceCount": int(
                (values != stripped_values).sum()
            ),
            "BlankCount": int(
                (stripped_values == "").sum()
            ),
            "MinimumLength": int(
                values.str.len().min()
            ) if len(values) > 0 else 0,
            "MaximumLength": int(
                values.str.len().max()
            ) if len(values) > 0 else 0
        }

    results["text"] = text_profile

    # -------------------------------------------------
    # 4. CATEGORICAL / BOOLEAN PROFILING
    # -------------------------------------------------

    categorical_types = [
        "categorical",
        "boolean"
    ]

    categorical_columns = detected_types[
        detected_types.isin(
            categorical_types
        )
    ].index

    categorical_profile = {}

    for column in categorical_columns:

        value_counts = df[
            column
        ].value_counts(dropna=False)

        categorical_profile[column] = {
            "UniqueCount": int(
                df[column].nunique(dropna=True)
            ),
            "MostCommonValue": (
                value_counts.index[0]
                if len(value_counts) > 0
                else None
            ),
            "MostCommonCount": (
                int(value_counts.iloc[0])
                if len(value_counts) > 0
                else 0
            ),
            "CategoryDistribution":
                value_counts.to_dict()
        }

    results["categorical"] = categorical_profile

    # -------------------------------------------------
    # 5. NUMERIC PROFILING
    # -------------------------------------------------

    numeric_columns = detected_types[
        detected_types == "numeric"
    ].index

    numeric_profile = {}

    for column in numeric_columns:

        values = df[column].dropna()

        numeric_profile[column] = {
            "Count": int(values.count()),
            "Minimum": values.min(),
            "Maximum": values.max(),
            "Mean": values.mean(),
            "Median": values.median(),
            "ZeroCount": int(
                (values == 0).sum()
            ),
            "NegativeCount": int(
                (values < 0).sum()
            )
        }

    results["numeric"] = numeric_profile

    # -------------------------------------------------
    # 6. DATE PROFILING
    # -------------------------------------------------

    date_columns = detected_types[
        detected_types == "date"
    ].index

    date_profile = {}

    for column in date_columns:

        raw_values = (
            df[column]
            .dropna()
            .astype(str)
            .str.strip()
        )

        parsed_iso = pd.to_datetime(
            raw_values,
            format="%Y-%m-%d",
            errors="coerce"
        )

        parsed_dmy = pd.to_datetime(
            raw_values,
            format="%d/%m/%Y",
            errors="coerce"
        )

        parsed_mdy = pd.to_datetime(
            raw_values,
            format="%m/%d/%Y",
            errors="coerce"
        )

        parsed_general = pd.to_datetime(
            raw_values,
            errors="coerce"
        )

        ambiguous_mask = (
            parsed_dmy.notna()
            & parsed_mdy.notna()
            & (
                parsed_dmy != parsed_mdy
            )
        )

        format_matches = pd.DataFrame({
            "ISO": parsed_iso.notna(),
            "DMY": parsed_dmy.notna(),
            "MDY": parsed_mdy.notna()
        })

        formats_used = format_matches.sum()

        date_profile[column] = {
            "ValidDateCount": int(
                parsed_general.notna().sum()
            ),
            "InvalidDateCount": int(
                parsed_general.isna().sum()
            ),
            "AmbiguousDateCount": int(
                ambiguous_mask.sum()
            ),
            "MixedFormat": bool(
                (formats_used > 0).sum() > 1
            ),
            "MinimumDate":
                parsed_general.min(),
            "MaximumDate":
                parsed_general.max(),
            "FutureDateCount": int(
                (
                    parsed_general
                    > pd.Timestamp.today().normalize()
                ).sum()
            )
        }

    results["date"] = date_profile

    # -------------------------------------------------
    # 7. PATTERN PROFILING
    # -------------------------------------------------

    email_profile = {}
    phone_profile = {}
    identifier_profile = {}

    # ---------- EMAIL ----------

    email_columns = detected_types[
        detected_types == "email"
    ].index

    email_pattern = (
        r"^[^@\s]+@[^@\s]+\.[^@\s]+$"
    )

    for column in email_columns:

        values = (
            df[column]
            .dropna()
            .astype(str)
            .str.strip()
        )

        email_matches = values.str.match(
            email_pattern,
            na=False
        )

        email_profile[column] = {
            "EmailLikeValueCount": int(
                email_matches.sum()
            ),
            "InvalidEmailLikeValueCount": int(
                (~email_matches).sum()
            ),
            "EmailPatternMatchRate": round(
                email_matches.mean() * 100,
                2
            ),
            "DuplicateEmailCount": int(
                values.duplicated().sum()
            )
        }

    # ---------- PHONE ----------

    phone_columns = detected_types[
        detected_types == "phone"
    ].index

    phone_pattern = (
        r"^\+?[0-9\s().-]{7,}$"
    )

    for column in phone_columns:

        values = (
            df[column]
            .dropna()
            .astype(str)
            .str.strip()
        )

        phone_matches = values.str.match(
            phone_pattern,
            na=False
        )

        phone_profile[column] = {
            "PhoneLikeValueCount": int(
                phone_matches.sum()
            ),
            "InvalidPhoneLikeValueCount":
                int((~phone_matches).sum()),
            "PhonePatternMatchRate": round(
                phone_matches.mean() * 100,
                2
            )
        }

    # ---------- IDENTIFIER ----------

    identifier_columns = detected_types[
        detected_types == "identifier"
    ].index

    for column in identifier_columns:

        values = (
            df[column]
            .dropna()
            .astype(str)
            .str.strip()
        )

        identifier_profile[column] = {
            "UniqueCount": int(
                values.nunique()
            ),
            "DuplicateCount": int(
                values.duplicated().sum()
            ),
            "MissingCount": int(
                df[column].isna().sum()
            )
        }

    results["patterns"] = {
        "email": email_profile,
        "phone": phone_profile,
        "identifier": identifier_profile
    }

    # -------------------------------------------------
    # 8. ISSUE / FLAG DETECTION
    # -------------------------------------------------

    issues = []

    # ---------- EXACT DUPLICATE ROWS ----------

    exact_duplicates = results[
        "general"
    ]["exact_duplicate_rows"]

    if exact_duplicates > 0:

        issues.append({
            "Column": None,
            "IssueType": "ExactDuplicateRows",
            "Severity": "High",
            "Count": exact_duplicates,
            "Description":
                "Exact duplicate rows detected."
        })

    # ---------- MISSING VALUES ----------

    for column, row in general_profile.iterrows():

        if row["MissingCount"] > 0:

            issues.append({
                "Column": column,
                "IssueType": "MissingValues",
                "Severity": "Medium",
                "Count": int(
                    row["MissingCount"]
                ),
                "Description":
                    "Missing values detected."
            })

    # ---------- TEXT ISSUES ----------

    for column, profile in text_profile.items():

        if profile["WhitespaceCount"] > 0:

            issues.append({
                "Column": column,
                "IssueType": "Whitespace",
                "Severity": "Low",
                "Count": profile[
                    "WhitespaceCount"
                ],
                "Description":
                    "Leading or trailing whitespace detected."
            })

        if profile["BlankCount"] > 0:

            issues.append({
                "Column": column,
                "IssueType": "BlankText",
                "Severity": "Medium",
                "Count": profile[
                    "BlankCount"
                ],
                "Description":
                    "Blank text values detected."
            })

    # ---------- DATE ISSUES ----------

    for column, profile in date_profile.items():

        if profile["InvalidDateCount"] > 0:

            issues.append({
                "Column": column,
                "IssueType": "InvalidDate",
                "Severity": "High",
                "Count": profile[
                    "InvalidDateCount"
                ],
                "Description":
                    "Values could not be interpreted as dates."
            })

        if profile["AmbiguousDateCount"] > 0:

            issues.append({
                "Column": column,
                "IssueType": "AmbiguousDate",
                "Severity": "High",
                "Count": profile[
                    "AmbiguousDateCount"
                ],
                "Description":
                    "Potentially ambiguous date values detected."
            })

        if profile["MixedFormat"]:

            issues.append({
                "Column": column,
                "IssueType": "MixedDateFormat",
                "Severity": "Medium",
                "Count": 0,
                "Description":
                    "Multiple recognized date formats detected."
            })

        if profile["FutureDateCount"] > 0:

            issues.append({
                "Column": column,
                "IssueType": "FutureDate",
                "Severity": "Medium",
                "Count": profile[
                    "FutureDateCount"
                ],
                "Description":
                    "Dates later than the current date detected."
            })

    # ---------- EMAIL ISSUES ----------

    for column, profile in email_profile.items():

        if profile[
            "InvalidEmailLikeValueCount"
        ] > 0:

            issues.append({
                "Column": column,
                "IssueType": "InvalidEmail",
                "Severity": "Medium",
                "Count": profile[
                    "InvalidEmailLikeValueCount"
                ],
                "Description":
                    "Values do not match the detected email pattern."
            })

        if profile[
            "DuplicateEmailCount"
        ] > 0:

            issues.append({
                "Column": column,
                "IssueType": "DuplicateEmail",
                "Severity": "Medium",
                "Count": profile[
                    "DuplicateEmailCount"
                ],
                "Description":
                    "Duplicate email values detected."
            })

    # ---------- PHONE ISSUES ----------

    for column, profile in phone_profile.items():

        if profile[
            "InvalidPhoneLikeValueCount"
        ] > 0:

            issues.append({
                "Column": column,
                "IssueType": "InvalidPhone",
                "Severity": "Medium",
                "Count": profile[
                    "InvalidPhoneLikeValueCount"
                ],
                "Description":
                    "Values do not match the detected phone pattern."
            })

    # ---------- IDENTIFIER ISSUES ----------

    for column, profile in identifier_profile.items():

        if profile["DuplicateCount"] > 0:

            issues.append({
                "Column": column,
                "IssueType": "DuplicateIdentifier",
                "Severity": "High",
                "Count": profile[
                    "DuplicateCount"
                ],
                "Description":
                    "Repeated identifier values detected."
            })

        if profile["MissingCount"] > 0:

            issues.append({
                "Column": column,
                "IssueType": "MissingIdentifier",
                "Severity": "High",
                "Count": profile[
                    "MissingCount"
                ],
                "Description":
                    "Missing identifier values detected."
            })

    # -------------------------------------------------
    # 9. FINAL ISSUE TABLE
    # -------------------------------------------------

    results["issues"] = pd.DataFrame(
        issues,
        columns=[
            "Column",
            "IssueType",
            "Severity",
            "Count",
            "Description"
        ]
    )

    # Sort issues by severity
    severity_order = {
        "High": 1,
        "Medium": 2,
        "Low": 3
    }

    results["issues"]["SeverityRank"] = (
        results["issues"]["Severity"]
        .map(severity_order)
    )

    results["issues"] = (
        results["issues"]
        .sort_values(
            "SeverityRank"
        )
        .drop(
            columns=["SeverityRank"]
        )
        .reset_index(drop=True)
    )

    # -------------------------------------------------
    # 10. CONFIGURATION
    # -------------------------------------------------

    results["configuration"] = config

    return results

In [ ]:
def detect_field_types(df):
    """
    Detect the likely semantic type of each column.

    Detection uses:
    - column-name signals
    - pandas data type
    - value patterns

    This is a detection aid, not a business decision engine.
    """

    field_types = {}

    for column in df.columns:

        name = column.lower()
        values = df[column].dropna().astype(str).str.strip()

        # ---------------------------------------------
        # 1. BOOLEAN
        # ---------------------------------------------

        if df[column].dtype == "bool":
            field_types[column] = "boolean"
            continue

        # ---------------------------------------------
        # 2. NUMERIC
        # ---------------------------------------------

        if pd.api.types.is_numeric_dtype(df[column]):
            field_types[column] = "numeric"
            continue

        # ---------------------------------------------
        # 3. EMAIL
        # ---------------------------------------------

        email_pattern = r"^[^@\s]+@[^@\s]+\.[^@\s]+$"

        if len(values) > 0:

            email_match_rate = values.str.match(
                email_pattern,
                na=False
            ).mean()

            if (
                "email" in name
                or "e-mail" in name
                or email_match_rate >= 0.8
            ):
                field_types[column] = "email"
                continue

        # ---------------------------------------------
        # 4. DATE / TIME
        # ---------------------------------------------

        date_name_signal = any(
            keyword in name
            for keyword in [
                "date",
                "datetime",
                "timestamp",
                "createdon",
                "modifiedon",
                "created_at",
                "updated_at",
                "time"
            ]
        )

        if date_name_signal and len(values) > 0:

            parsed_dates = pd.to_datetime(
                values,
                errors="coerce"
            )

            date_match_rate = parsed_dates.notna().mean()

            if date_match_rate >= 0.8:
                field_types[column] = "date"
                continue

        # ---------------------------------------------
        # 5. PHONE
        # ---------------------------------------------

        phone_pattern = r"^\+?[0-9\s().-]{7,}$"

        phone_name_signal = any(
            keyword in name
            for keyword in [
                "phone",
                "mobile",
                "telephone",
                "tel"
            ]
        )

        if len(values) > 0:

            phone_match_rate = values.str.match(
                phone_pattern,
                na=False
            ).mean()

            if (
                phone_name_signal
                or phone_match_rate >= 0.8
            ):
                field_types[column] = "phone"
                continue

        # ---------------------------------------------
        # 6. IDENTIFIER
        # ---------------------------------------------

        identifier_name_signal = any(
            keyword in name
            for keyword in [
                "id",
                "code",
                "key"
            ]
        )

        if identifier_name_signal:
            field_types[column] = "identifier"
            continue

        # ---------------------------------------------
        # 7. CATEGORICAL / TEXT
        # ---------------------------------------------

        unique_ratio = (
            df[column].nunique(dropna=True)
            / len(df)
            if len(df) > 0
            else 0
        )

        if unique_ratio <= 0.20:
            field_types[column] = "categorical"
        else:
            field_types[column] = "text"

    return pd.Series(field_types, name="DetectedFieldType")

In [ ]:
detected_types = detect_field_types(supplier_raw)

detected_types

In [ ]:
profile_results["general"]["column_profile"]

In [ ]:
profile_results["text"]

In [ ]:
profile_results["categorical"]

In [ ]:
profile_results["numeric"]

In [ ]:
profile_results["date"]

In [35]:
profile_results["patterns"]

{'email': {'EmailAddress': {'EmailLikeValueCount': 9,
   'InvalidEmailLikeValueCount': 0,
   'EmailPatternMatchRate': np.float64(100.0),
   'DuplicateEmailCount': 1}},
 'phone': {'PhoneNumber': {'PhoneLikeValueCount': 9,
   'InvalidPhoneLikeValueCount': 0,
   'PhonePatternMatchRate': np.float64(100.0),
   'DuplicatePhoneCount': 1}},
 'identifier': {'SupplierCode': {'UniqueCount': 8,
   'DuplicateCount': 1,
   'MissingCount': 0}}}

In [ ]:
profile_results["field_types"]

In [34]:
profile_results["issues"]

,Column,IssueType,Severity,Count,Description
0,None,ExactDuplicateRows,High,1,Exact duplicate rows detected.
1,SupplierCode,DuplicateIdentifier,High,1,Repeated identifier values detected.
2,EmailAddress,DuplicateEmail,Medium,1,Duplicate email values detected.
3,City,Whitespace,Low,2,Leading or trailing whitespace detected.
4,SupplierName,Whitespace,Low,1,Leading or trailing whitespace detected.


In [ ]:
test_data = pd.DataFrame({
    "Field_17": [
        "2026-07-01",
        "2026-07-02",
        "2026-07-03",
        "2026-07-04"
    ]
})

test_result = profiler.profile_dataset(test_data)

test_result["field_types"]

In [ ]:
test_data = pd.DataFrame({
    "UnknownColumn": [
        "2026-07-01",
        "02/07/2026",
        "2026-07-03",
        "04/07/2026"
    ]
})

test_result = profiler.profile_dataset(test_data)

print(test_result["field_types"])
print(test_result["date"])

In [ ]:
test_data = pd.DataFrame({
    "UnknownColumn": [
        "2026-07-01",
        "2026-99-99",
        "not-a-date",
        "2026-02-30"
    ]
})

test_result = profiler.profile_dataset(test_data)

print(test_result["field_types"])
print(test_result["date"])

In [ ]:
test_data = pd.DataFrame({
    "UnknownColumn": [
        "2026-07-01",
        "2026-07-02",
        "2026-07-03",
        "2026-07-04",
        "2026-07-05",
        "2026-07-06",
        "2026-07-07",
        "2026-07-08",
        "2026-99-99",
        "02/07/2026"
    ]
})

test_result = profiler.profile_dataset(test_data)

print(test_result["field_types"])
print(test_result["date"])

In [ ]:
test_data = pd.DataFrame({
    "Column_A": [
        "john@example.com",
        "mary@example.com",
        "test@example.com",
        "admin@example.com"
    ],

    "Column_B": [
        "+971501234567",
        "+971502345678",
        "+971503456789",
        "+971504567890"
    ],

    "Column_C": [
        "SUP001",
        "SUP002",
        "SUP003",
        "SUP004"
    ]
})

test_result = profiler.profile_dataset(test_data)

print(test_result["field_types"])
print(test_result["patterns"])

In [ ]:
test_data = pd.DataFrame({
    "UnknownColumn": [
        "ABC0001",
        "ABC0002",
        "ABC0003",
        "ABC0004",
        "ABC0005"
    ]
})

test_result = profiler.profile_dataset(test_data)

print(test_result["field_types"])
print(test_result["patterns"])

In [ ]:
test_data = pd.DataFrame({
    "UnknownColumn": [
        "1250.50",
        "2300.00",
        "875.75",
        "4100.25",
        "999.99"
    ]
})

test_result = profiler.profile_dataset(test_data)

print(test_result["field_types"])
print(test_result["numeric"])

In [ ]:
test_data = pd.DataFrame({
    "UnknownAmount": [
        "1250.50",
        "2300.00",
        "875.75",
        "4100.25",
        "999.99",
        "N/A"
    ]
})

test_result = profiler.profile_dataset(test_data)

print(test_result["field_types"])
print(test_result["numeric"])

In [ ]:
test_data = pd.DataFrame({
    "PlainNumber": [
        "1250.50",
        "2300.00",
        "875.75",
        "4100.25"
    ],

    "ThousandsComma": [
        "1,250.50",
        "2,300.00",
        "875.75",
        "4,100.25"
    ],

    "CurrencyAED": [
        "AED 1,250.50",
        "AED 2,300.00",
        "AED 875.75",
        "AED 4,100.25"
    ],

    "CurrencySymbol": [
        "$1,250.50",
        "$2,300.00",
        "$875.75",
        "$4,100.25"
    ],

    "Accounting": [
        "(1,250.50)",
        "2,300.00",
        "(875.75)",
        "4,100.25"
    ],

    "IdentifierLike": [
        "001234",
        "001235",
        "001236",
        "001237"
    ]
})

test_result = profiler.profile_dataset(test_data)

print(test_result["field_types"])
print(test_result["numeric"])

In [ ]:
test_data = pd.DataFrame({
    "Percentage": [
        "15%",
        "12.5%",
        "7%",
        "100%"
    ],

    "NegativeNumber": [
        "-1250.50",
        "-2300.00",
        "-875.75",
        "-4100.25"
    ],

    "CurrencyNegative": [
        "-AED 1,250.50",
        "AED -2,300.00",
        "(AED 875.75)",
        "(AED 4,100.25)"
    ]
})

test_result = profiler.profile_dataset(test_data)

print(test_result["field_types"])
print(test_result["numeric"])

In [ ]:
test_data = pd.DataFrame({
    "NegativeNumber": [
        "-1250.50",
        "-2300.00",
        "-875.75",
        "-4100.25"
    ]
})

test_result = profiler.profile_dataset(test_data)

print(test_result["field_types"])
print(test_result["numeric"])

In [ ]:
test_data = pd.DataFrame({
    "PlainNumber": [
        "1250.50",
        "2300.00",
        "875.75",
        "4100.25"
    ],

    "ThousandsComma": [
        "1,250.50",
        "2,300.00",
        "875.75",
        "4,100.25"
    ],

    "CurrencyAED": [
        "AED 1,250.50",
        "AED 2,300.00",
        "AED 875.75",
        "AED 4,100.25"
    ],

    "CurrencySymbol": [
        "$1,250.50",
        "$2,300.00",
        "$875.75",
        "$4,100.25"
    ],

    "Accounting": [
        "(1,250.50)",
        "2,300.00",
        "(875.75)",
        "4,100.25"
    ],

    "Percentage": [
        "15%",
        "12.5%",
        "7%",
        "100%"
    ],

    "IdentifierLike": [
        "001234",
        "001235",
        "001236",
        "001237"
    ]
})

test_result = profiler.profile_dataset(test_data)

print(test_result["field_types"])
print(test_result["numeric"])

In [ ]:
import profiler.data_profiler_v1 as profiler

print(profiler.__file__)
print(profiler.DEFAULT_CONFIG)

In [ ]:
import profiler.data_profiler_v1 as profiler

print(profiler.DEFAULT_CONFIG)

In [ ]:
test_data = pd.DataFrame({
    "MixedAmount": [
        "1,250.50",
        "AED 2,300.00",
        "N/A",
        "4,100.25",
        "(875.75)"
    ]
})

test_result = profiler.profile_dataset(test_data)

print(test_result["field_types"])
print(test_result["numeric"])

In [ ]:
test_data = pd.DataFrame({
    "Revenue": [
        "1250.50",
        "2300.00",
        "875.75",
        "INVALID",
        "4100.25",
        "-500.00",
        "0"
    ]
})

test_result = profiler.profile_dataset(test_data)

print(test_result["field_types"])
print(test_result["numeric"])

In [ ]:
test_data = pd.DataFrame({
    "CustomerID": [
        "10001",
        "10002",
        "10003",
        "10004",
        "10005"
    ],

    "OrderID": [
        "500001",
        "500002",
        "500003",
        "500004",
        "500005"
    ],

    "Quantity": [
        "10",
        "25",
        "5",
        "12",
        "8"
    ]
})

test_result = profiler.profile_dataset(test_data)

print(test_result["field_types"])

In [ ]:
test_data = pd.DataFrame({
    "Amount": [
        "ABC",
        "DEF",
        "GHI",
        "JKL"
    ],

    "Code": [
        "1001",
        "1002",
        "1003",
        "1004"
    ],

    "UnknownColumn": [
        "2026-07-01",
        "2026-07-02",
        "2026-07-03",
        "2026-07-04"
    ]
})

test_result = profiler.profile_dataset(test_data)

print(test_result["field_types"])

In [32]:
test_data = pd.DataFrame({
    "CustomerName": [
        "John",
        None,
        "",
        "   ",
        "N/A",
        "Unknown",
        "Mary"
    ]
})

test_result = profiler.profile_dataset(test_data)

print(test_result.keys())

dict_keys(['field_types', 'detection_details', 'general', 'text', 'categorical', 'numeric', 'date', 'patterns', 'issues', 'configuration'])


In [33]:
print(test_result["general"])

{'rows': 7, 'columns': 1, 'exact_duplicate_rows': 0, 'column_profile':              DataType DetectedFieldType  MissingCount  MissingPercent  \
CustomerName   object              text             1           14.29   

              UniqueCount  DuplicateValueCount  
CustomerName            6                    0  }


In [36]:
test_data = pd.DataFrame({
    "CustomerName": [
        "John",
        None,
        "",
        "   ",
        "N/A",
        "Unknown",
        "Mary"
    ]
})

test_result = profiler.profile_dataset(test_data)

print(test_data)
print()
print(test_result["general"])
print()
print(test_result["issues"])
print()
print(test_result["patterns"])

  CustomerName
0         John
1         None
2             
3             
4          N/A
5      Unknown
6         Mary

{'rows': 7, 'columns': 1, 'exact_duplicate_rows': 0, 'column_profile':              DataType DetectedFieldType  MissingCount  MissingPercent  \
CustomerName   object              text             1           14.29   

              UniqueCount  DuplicateValueCount  
CustomerName            6                    0  }

         Column      IssueType Severity  Count  \
0  CustomerName  MissingValues   Medium      1   
1  CustomerName      BlankText   Medium      2   
2  CustomerName     Whitespace      Low      1   

                                Description  
0                  Missing values detected.  
1               Blank text values detected.  
2  Leading or trailing whitespace detected.  

{'email': {}, 'phone': {}, 'identifier': {}}


In [37]:
test_data = pd.DataFrame({
    "TestValues": [
        None,
        float("nan"),
        "",
        "   ",
        "N/A",
        "NA",
        "NULL",
        "null",
        "Unknown",
        "Valid Value"
    ]
})

test_result = profiler.profile_dataset(test_data)

print(test_result["general"])
print()
print(test_result["issues"])

{'rows': 10, 'columns': 1, 'exact_duplicate_rows': 0, 'column_profile':            DataType DetectedFieldType  MissingCount  MissingPercent  \
TestValues   object              text             2            20.0   

            UniqueCount  DuplicateValueCount  
TestValues            8                    0  }

       Column      IssueType Severity  Count  \
0  TestValues  MissingValues   Medium      2   
1  TestValues      BlankText   Medium      2   
2  TestValues     Whitespace      Low      1   

                                Description  
0                  Missing values detected.  
1               Blank text values detected.  
2  Leading or trailing whitespace detected.  


In [41]:
test_data = pd.DataFrame({
    "TestValues": [
        None,
        float("nan"),
        "",
        "   ",
        "N/A",
        "NA",
        "NULL",
        "null",
        "Unknown",
        "Valid Value"
    ]
})

test_result = profiler.profile_dataset(test_data)

print(test_result["text"])
print()
print(test_result["issues"])

{'TestValues': {'WhitespaceCount': 1, 'BlankCount': 2, 'PotentialMissingMarkerCount': 5, 'PotentialMissingMarkers': ['n/a', 'na', 'null', 'unknown'], 'MinimumLength': 0, 'MaximumLength': 11, 'AverageLength': np.float64(3.88)}}

       Column               IssueType Severity  Count  \
0  TestValues           MissingValues   Medium      2   
1  TestValues               BlankText   Medium      2   
2  TestValues              Whitespace      Low      1   
3  TestValues  PotentialMissingMarker      Low      5   

                                         Description  
0                           Missing values detected.  
1                        Blank text values detected.  
2           Leading or trailing whitespace detected.  
3  Potential missing-value markers detected; revi...  


In [42]:
profile_results = profiler.profile_dataset(supplier_raw)

print(profile_results["text"])
print()
print(profile_results["issues"])

{'SupplierName': {'WhitespaceCount': 1, 'BlankCount': 0, 'PotentialMissingMarkerCount': 0, 'PotentialMissingMarkers': [], 'MinimumLength': 15, 'MaximumLength': 25, 'AverageLength': np.float64(20.78)}, 'ContactPerson': {'WhitespaceCount': 0, 'BlankCount': 0, 'PotentialMissingMarkerCount': 0, 'PotentialMissingMarkers': [], 'MinimumLength': 6, 'MaximumLength': 12, 'AverageLength': np.float64(9.89)}, 'StreetAddress': {'WhitespaceCount': 0, 'BlankCount': 0, 'PotentialMissingMarkerCount': 0, 'PotentialMissingMarkers': [], 'MinimumLength': 5, 'MaximumLength': 23, 'AverageLength': np.float64(12.78)}, 'City': {'WhitespaceCount': 2, 'BlankCount': 0, 'PotentialMissingMarkerCount': 0, 'PotentialMissingMarkers': [], 'MinimumLength': 5, 'MaximumLength': 9, 'AverageLength': np.float64(5.89)}}

         Column            IssueType Severity  Count  \
0          None   ExactDuplicateRows     High      1   
1  SupplierCode  DuplicateIdentifier     High      1   
2  EmailAddress       DuplicateEmail   Med

In [43]:
print(profile_results["general"]["column_profile"])

               DataType DetectedFieldType  MissingCount  MissingPercent  \
SupplierCode     object        identifier             0             0.0   
SupplierName     object              text             0             0.0   
ContactPerson    object              text             0             0.0   
EmailAddress     object             email             0             0.0   
PhoneNumber      object             phone             0             0.0   
StreetAddress    object              text             0             0.0   
City             object              text             0             0.0   
Country          object       categorical             0             0.0   
ActiveFlag         bool           boolean             0             0.0   
CreatedOn        object              date             0             0.0   
CreatedByUser    object       categorical             0             0.0   
ModifiedOn       object              date             0             0.0   
ModifiedByUser   object  

In [44]:
test_data = pd.DataFrame({
    "Revenue": [
        100,
        110,
        105,
        98,
        102,
        101,
        99,
        5000
    ]
})

test_result = profiler.profile_dataset(test_data)

print(test_result["numeric"])

{'Revenue': {'ValidNumericCount': 8, 'InvalidNumericCount': 0, 'Minimum': np.float64(98.0), 'Maximum': np.float64(5000.0), 'Mean': np.float64(714.375), 'Median': np.float64(101.5), 'ZeroCount': 0, 'NegativeCount': 0, 'PositiveCount': 8, 'RecognizedRepresentations': ['Integer'], 'CurrencyIndicators': [], 'PercentageCount': 0, 'AccountingCount': 0, 'OutlierCandidateCount': 1, 'LowerOutlierBoundary': np.float64(90.0), 'UpperOutlierBoundary': np.float64(116.0)}}


In [45]:
profile_results = profiler.profile_dataset(supplier_raw)

print(profile_results["field_types"])
print(profile_results["general"])
print(profile_results["text"])
print(profile_results["numeric"])
print(profile_results["date"])
print(profile_results["issues"])

SupplierCode       identifier
SupplierName             text
ContactPerson            text
EmailAddress            email
PhoneNumber             phone
StreetAddress            text
City                     text
Country           categorical
ActiveFlag            boolean
CreatedOn                date
CreatedByUser     categorical
ModifiedOn               date
ModifiedByUser    categorical
Name: DetectedFieldType, dtype: object
{'rows': 9, 'columns': 13, 'exact_duplicate_rows': 1, 'column_profile':                DataType DetectedFieldType  MissingCount  MissingPercent  \
SupplierCode     object        identifier             0             0.0   
SupplierName     object              text             0             0.0   
ContactPerson    object              text             0             0.0   
EmailAddress     object             email             0             0.0   
PhoneNumber      object             phone             0             0.0   
StreetAddress    object              text        